<a href="https://colab.research.google.com/github/bmanibharathibe/trendpulse-manibharathi/blob/main/Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import requests
import json
import os
import time
from datetime import datetime

headers = {"User-Agent": "TrendPulse/1.0"}

TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

categories = {
    "technology": ["ai","software","tech","code","computer","data","cloud","api","gpu","llm"],
    "worldnews": ["war","government","country","president","election","climate","attack","global"],
    "sports": ["nfl","nba","fifa","sport","game","team","player","league","championship"],
    "science": ["research","study","space","physics","biology","discovery","nasa","genome"],
    "entertainment": ["movie","film","music","netflix","game","book","show","award","streaming"]
}

collected_stories = []
category_counts = {cat: 0 for cat in categories}

# Fetch top story IDs
try:
    response = requests.get(TOP_STORIES_URL, headers=headers)
    story_ids = response.json()
except:
    print("Error fetching top stories")
    story_ids = []

# Function to detect category
def detect_category(title):
    title = title.lower()
    for cat, words in categories.items():
        for word in words:
            if word in title:
                return cat
    return None

# Loop through categories
for category in categories:

    print(f"Collecting stories for {category}")

    for story_id in story_ids:

        if category_counts[category] >= 35:
            break

        try:
            r = requests.get(ITEM_URL.format(story_id), headers=headers)
            story = r.json()
        except:
            print("Request failed for", story_id)
            continue

        if not story or "title" not in story:
            continue

        detected = detect_category(story["title"])

        if detected != category:
            continue

        story_data = {
            "post_id": story.get("id"),
            "title": story.get("title"),
            "category": category,
            "score": story.get("score", 0),
            "num_comments": story.get("descendants", 0),
            "author": story.get("by"),
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        collected_stories.append(story_data)
        category_counts[category] += 1

    # sleep after each category
    time.sleep(2)

# Create data folder
if not os.path.exists("data"):
    os.makedirs("data")

filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

with open(filename, "w") as f:
    json.dump(collected_stories, f, indent=4)

print(f"Collected {len(collected_stories)} stories. Saved to {filename}")

Collected 112 stories. Saved to data/trends_20260408.json
